In [ ]:
# Cell 1 — Install dependencies (run once)
# !pip install rasterio geopandas matplotlib numpy

# Cell 2 — Imports
import rasterio
from rasterio.plot import show
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Cell 3 — File paths (update these)
TIF_PATH  = "D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\SIT\sit_om16.tif"   # ← your orthomosaic
GPKG_PATH = "D:\gaurav4\Drone-Phenology-Monitoring-main_new\Drone-Phenology-Monitoring-main\orthomosaics\output\input_crowns_multithreshold_lhc_10Mar26\OM16_multithreshold.gpkg"       # ← your detected crowns

# Cell 4 — Load data
crowns = gpd.read_file(GPKG_PATH)
print(f"Crowns CRS : {crowns.crs}")
print(f"Crowns count: {len(crowns)}")

with rasterio.open(TIF_PATH) as src:
    print(f"Raster CRS : {src.crs}")
    print(f"Raster size: {src.width} x {src.height}")
    print(f"Raster bands: {src.count}")

# Cell 5 — Reproject crowns to match raster CRS (if needed)
with rasterio.open(TIF_PATH) as src:
    raster_crs = src.crs

if crowns.crs != raster_crs:
    print(f"Reprojecting crowns from {crowns.crs} → {raster_crs}")
    crowns = crowns.to_crs(raster_crs)
else:
    print("CRS match — no reprojection needed")

# Cell 6 — Plot overlay
fig, ax = plt.subplots(figsize=(14, 14))

with rasterio.open(TIF_PATH) as src:
    # Read RGB bands (assumes bands 1,2,3 are R,G,B — adjust if needed)
    if src.count >= 3:
        img = src.read([1, 2, 3])
        # Normalize to 0–1 for display
        img = img.astype(float)
        for i in range(3):
            band = img[i]
            p2, p98 = np.percentile(band[band > 0], (2, 98))
            img[i] = np.clip((band - p2) / (p98 - p2), 0, 1)
        show(img, transform=src.transform, ax=ax)
    else:
        # Single-band (e.g. DSM / grayscale)
        show(src, ax=ax, cmap="gray")

# Overlay crowns — outline only, transparent fill
crowns.plot(
    ax=ax,
    facecolor="none",        # transparent fill
    edgecolor="red",         # crown outline colour
    linewidth=0.8,
    alpha=0.9
)

ax.set_title("Orthomosaic with Detected Tree Crowns", fontsize=16)
ax.set_xlabel("Easting")
ax.set_ylabel("Northing")
ax.axis("equal")

legend_patch = mpatches.Patch(facecolor="none", edgecolor="red", linewidth=1.5,
                               label=f"Tree Crowns (n={len(crowns)})")
ax.legend(handles=[legend_patch], loc="upper right", fontsize=11)

plt.tight_layout()
plt.savefig("overlay_output.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved → overlay_output.png")